## ML_1037: ViT Patch Embedding

**Difficulty**: Medium | **TorchCode**: #27

### Problem
Implement Vision Transformer patch embedding. Split an image into non-overlapping patches, flatten each patch, and project to `embed_dim` with a linear layer.

### Formula
$$N_{\text{patches}} = \left(\frac{H}{P}\right)^2, \quad z = \text{reshape}(x, [B, N, C \cdot P^2]) \cdot W_E$$

In [1]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Linear(in_channels * patch_size * patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        n_h, n_w = H // p, W // p
        x = x.reshape(B, C, n_h, p, n_w, p)
        x = x.permute(0, 2, 4, 1, 3, 5).reshape(B, n_h * n_w, C * p * p)
        return self.proj(x)

/home/kenneth/CODE/AI_PROJECTS/Leetcode/.venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)                  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2) # (B, N, embed_dim)
        return x


In [3]:
import torch
import torch.nn as nn

# --- Test 1: Output shape ---
pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=64)
assert isinstance(pe, nn.Module)
assert pe(torch.randn(2, 3, 32, 32)).shape == (2, 16, 64)

# --- Test 2: num_patches attribute ---
pe2 = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
assert pe2.num_patches == 196

# --- Test 3: Different image sizes ---
pe3 = PatchEmbedding(img_size=64, patch_size=16, in_channels=1, embed_dim=32)
assert pe3(torch.randn(1, 1, 64, 64)).shape == (1, 16, 32)

# --- Test 4: Gradient flow ---
x = torch.randn(1, 3, 32, 32, requires_grad=True)
pe(x).sum().backward()
assert x.grad is not None

print('All tests passed!')

All tests passed!
